# Grand Conquest — `irotem98/Codex-Pets-Sprite-Sheet-Generator` Colab Runner

Runs Hugging Face's purpose-trained **`irotem98/Codex-Pets-Sprite-Sheet-Generator`** (`Boogu-Image-0.1-Edit` + alpha-compatible RGBA VAE + sprite-sheet LoRA) directly on Colab GPU.

### Workflow
1. **Alpha Cleanup:** Automatically strips baked semi-transparent spit mist / smoke halos from `static_all_54/<sprite_key>.png` before feeding it in.
2. **Sprite Sheet Diffusion:** Generates transparent `192x208` frame rows (`idle`, `run right`, `jump`, etc.) directly as native RGBA.
3. **Game Packing:** Extracts 4 progressive `run right` Walk frames + 4 action (`jump`/`leap`) frames cleanly packed into `512x256` RGBA `sheets_output/<sprite_key>.png`.

In [ ]:
#@title 1. Check GPU, Clone Codex-Pets-Sprite-Sheet-Generator & Unpack Sprite Kit
import os, shutil, zipfile, torch
assert torch.cuda.is_available(), 'No GPU detected! Click Runtime -> Change runtime type -> choose L4 or A100 (or T4 High-RAM).'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)')

REPO_DIR = '/content/Codex-Pets-Sprite-Sheet-Generator'
if not os.path.isdir(REPO_DIR):
    !git lfs install
    !git clone https://huggingface.co/irotem98/Codex-Pets-Sprite-Sheet-Generator {REPO_DIR}
!pip install -r {REPO_DIR}/requirements.txt
!pip uninstall -y torchao
!pip install -U opencv-python-headless pillow numpy
!python {REPO_DIR}/generate_sprite_sheet.py --help || true

from google.colab import drive
drive.mount('/content/drive')

DRIVE_KIT_DIR = '/content/drive/MyDrive/Colab Notebooks/grand-conquest-sprites'  #@param {type:"string"}
WORK_DIR = '/content/grand_conquest_work'
SHEETS_DRIVE_DIR = os.path.join(DRIVE_KIT_DIR, 'sheets_output')
RAW_DRIVE_DIR = os.path.join(DRIVE_KIT_DIR, 'raw_sheets')
CLEAN_DRIVE_DIR = os.path.join(DRIVE_KIT_DIR, 'clean_refs')
for d in (WORK_DIR, SHEETS_DRIVE_DIR, RAW_DRIVE_DIR, CLEAN_DRIVE_DIR):
    os.makedirs(d, exist_ok=True)

kit_zip = os.path.join(DRIVE_KIT_DIR, 'grand_conquest_sprite_kit.zip')
assert os.path.isfile(kit_zip), f'Missing {kit_zip}'
with zipfile.ZipFile(kit_zip, 'r') as zf:
    zf.extractall(WORK_DIR)

local_sheets = os.path.join(WORK_DIR, 'assets', 'sheets')
os.makedirs(local_sheets, exist_ok=True)
for fn in os.listdir(SHEETS_DRIVE_DIR):
    if fn.endswith('.png'):
        shutil.copy2(os.path.join(SHEETS_DRIVE_DIR, fn), os.path.join(local_sheets, fn))

%cd {WORK_DIR}
!python tools/slice_sprite_sheets.py --audit

In [ ]:
#@title 2. Strip Baked Mist/Sparks & Run Codex-Pets-Sprite-Sheet-Generator (Default: hive_acid_spitter)
import importlib, os, shutil, subprocess, sys, zipfile
import cv2
import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw

# Refresh tools & manifest from Drive zip so cached runtimes always get latest slicer
kit_zip = os.path.join(DRIVE_KIT_DIR, 'grand_conquest_sprite_kit.zip')
if os.path.isfile(kit_zip):
    with zipfile.ZipFile(kit_zip, 'r') as zf:
        zf.extractall(WORK_DIR)

if os.path.join(WORK_DIR, 'tools') not in sys.path:
    sys.path.insert(0, os.path.join(WORK_DIR, 'tools'))
import slice_sprite_sheets as slicer
importlib.reload(slicer)
slicer.REPO_ROOT = WORK_DIR
slicer.OUT_DIR = os.path.join(WORK_DIR, 'assets', 'sheets')
slicer.PROMPTS_PATH = os.path.join(WORK_DIR, 'tools', 'unit_sheet_prompts.json')

TARGET = 'hive_acid_spitter'  #@param ['hive_acid_spitter', 'FACTION:Hive', 'ALL_MISSING']
ROW_BATCH_SIZE = 1  #@param [1, 2, 4] {type:"raw"}
ACTION_ROW = 'jump'  #@param ['jump', 'wave']
FAST_2ROW_MODE = True  #@param {type:"boolean"}
REUSE_EXISTING_ATLAS = True  #@param {type:"boolean"}
OVERWRITE_EXISTING = True  #@param {type:"boolean"}

def clean_static_rgba(src_png, out_png):
    """Remove semi-transparent spray/mist/smoke halos while keeping solid inked sprite intact."""
    im = np.array(Image.open(src_png).convert('RGBA'))
    a = im[:, :, 3]
    solid = (a >= 195).astype(np.uint8)
    n, lbl, st, _ = cv2.connectedComponentsWithStats(solid, 8)
    if n > 1:
        areas = st[1:, cv2.CC_STAT_AREA]
        max_a = areas.max()
        keep_solid = np.zeros_like(solid)
        for idx, area in enumerate(areas, start=1):
            if area >= max(40, int(max_a * 0.08)):
                keep_solid[lbl == idx] = 1
    else:
        keep_solid = solid
    edge = cv2.dilate(keep_solid, np.ones((3, 3), np.uint8), iterations=1)
    clean_a = np.where(keep_solid > 0, 255, np.where((edge > 0) & (a > 70), a, 0)).astype(np.uint8)
    out = im.copy()
    out[:, :, 3] = clean_a
    Image.fromarray(out).save(out_png)
    return out_png

def preview_on_checker(img_or_path, max_w=768):
    fg = Image.open(img_or_path).convert('RGBA') if isinstance(img_or_path, str) else img_or_path.convert('RGBA')
    if fg.width > max_w:
        fg = fg.resize((max_w, int(fg.height * (max_w / fg.width))), Image.Resampling.LANCZOS)
    bg = Image.new('RGBA', fg.size, (38, 38, 42, 255))
    draw = ImageDraw.Draw(bg)
    for y in range(0, fg.height, 16):
        for x in range(0, fg.width, 16):
            if ((x // 16) + (y // 16)) % 2 == 0:
                draw.rectangle([x, y, x + 15, y + 15], fill=(54, 54, 60, 255))
    return Image.alpha_composite(bg, fg).convert('RGB')

def extract_8_from_codex_atlas(atlas_path, out_sheet, action_row='jump'):
    """Extract 4 Run-Right walk frames (Row 1) + 4 Action frames (Row 4 jump or Row 3 wave) into 512x256 RGBA."""
    atlas = np.array(Image.open(atlas_path).convert('RGBA'))
    cell_w = atlas.shape[1] // 8  # 192 px in 1536x2288 V2 atlas
    cell_h = 208 if atlas.shape[0] % 208 == 0 else atlas.shape[0] // 11

    def non_empty_cells(row_idx):
        y0, y1 = row_idx * cell_h, (row_idx + 1) * cell_h
        cells = []
        for c in range(8):
            cell = atlas[y0:y1, c * cell_w : (c + 1) * cell_w]
            if np.count_nonzero(cell[:, :, 3] > 20) > 40:
                cells.append(cell)
        return cells

    walk_row = non_empty_cells(1) or non_empty_cells(7) or non_empty_cells(0)  # Row 1: run right
    if action_row == 'wave':
        act_row = non_empty_cells(3) or non_empty_cells(4) or non_empty_cells(5)
    else:
        act_row = non_empty_cells(4) or non_empty_cells(3) or non_empty_cells(5)
    def pick4(cells):
        idxs = np.linspace(0, len(cells) - 1, 4).round().astype(int)
        return [cells[j] for j in idxs]

    frames_8 = pick4(walk_row) + pick4(act_row)
    slicer._pack_8_frames_rgba(frames_8, out_sheet, frame_sz=128)

manifest = slicer.load_unit_manifest()
selected = [
    (k, v) for k, v in manifest.items()
    if TARGET == 'ALL_MISSING'
    or (TARGET.startswith('FACTION:') and v['faction'].lower() == TARGET.split(':', 1)[1].lower())
    or k == TARGET
]
assert selected, f'No unit matched TARGET={TARGET}'

LOW_VRAM_RUNNER = '/content/run_codex_low_vram.py'
LOW_VRAM_CODE = '''
import gc
import os
import sys
from pathlib import Path

import torch
from PIL import Image

sys.path.insert(0, os.getcwd())
import generate_sprite_sheet as gs

def _move_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)
    if isinstance(obj, (list, tuple)):
        return type(obj)(_move_to_device(x, device) for x in obj)
    return obj

def _enable_block_offload(transformer, device):
    from accelerate import cpu_offload
    transformer.to('cpu')
    gc.collect()
    torch.cuda.empty_cache()
    for p in transformer.parameters(recurse=False):
        p.data = p.data.to(device)
    for b in transformer.buffers(recurse=False):
        b.data = b.data.to(device)
    for _, child in transformer.named_children():
        if isinstance(child, torch.nn.ModuleList):
            for block in child:
                cpu_offload(block, execution_device=device, offload_buffers=True)
        else:
            cpu_offload(child, execution_device=device, offload_buffers=True)

def _pad_and_cat(tensors, dim=0):
    if len(tensors) == 1:
        return tensors[0]
    max_len = max(t.shape[1] for t in tensors)
    padded = []
    for t in tensors:
        if t.shape[1] < max_len:
            pad_shape = list(t.shape)
            pad_shape[1] = max_len - t.shape[1]
            t = torch.cat([t, torch.zeros(pad_shape, dtype=t.dtype, device=t.device)], dim=1)
        padded.append(t)
    return torch.cat(padded, dim=dim)

def main():
    args = gs.parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is required for Boogu 10B')
    device = torch.device(args.device)
    total_gib = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    rows_env = os.environ.get('CODEX_ROWS', '').strip()
    target_rows = [int(x) for x in rows_env.split(',') if x.strip()] if rows_env else list(range(len(gs.ROW_FRAME_COUNTS)))
    print(f'[low-vram] Detected GPU with {total_gib:.2f} GiB VRAM | target rows: {target_rows}', flush=True)
    if total_gib >= 36.0 and not rows_env:
        gs.main()
        return

    dtype = torch.bfloat16
    release_root = Path(gs.__file__).resolve().parent
    compat = release_root / 'compat_no_transformer_engine'
    if compat.is_dir():
        sys.path.insert(0, str(compat))
    os.environ['device'] = args.device

    from diffusers import AutoencoderKL
    from transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor
    from boogu.models.transformers.transformer_boogu import BooguImageTransformer2DModel
    from boogu.pipelines.boogu.pipeline_boogu import BooguImagePipeline
    from boogu.schedulers.scheduling_flow_match_euler_discrete_time_shifting import FlowMatchEulerDiscreteScheduler

    with Image.open(args.input) as source:
        cell = gs.trim_and_fit_rgba(source)

    print('[low-vram] Stage 1/3: Pre-encoding all 11 row prompts with Qwen3-VL (then unloading MLLM from VRAM)...', flush=True)
    processor = Qwen3VLProcessor.from_pretrained(args.base_model, subfolder='processor')
    mllm = Qwen3VLForConditionalGeneration.from_pretrained(args.base_model, subfolder='mllm', torch_dtype=dtype)
    mllm.eval()
    for p in mllm.parameters():
        p.requires_grad_(False)
    if total_gib >= 18.0:
        mllm.to(device=device, dtype=dtype)
    else:
        _enable_block_offload(mllm, device)

    enc_pipe = BooguImagePipeline(transformer=None, vae=None, scheduler=None, mllm=mllm, processor=processor)
    import types
    tx_cfg = BooguImageTransformer2DModel.load_config(args.base_model, subfolder='transformer')
    enc_pipe.transformer = types.SimpleNamespace(
        instruction_feature_configs=tx_cfg.get('instruction_feature_configs', {'num_instruction_feature_layers': 1}),
        dtype=dtype,
    )
    enc_pipe.text_instruction_rewriter = None
    enc_pipe.instruction_rewriter_processor = None
    neg_emb, neg_mask = gs.encode_instruction(enc_pipe, [''], None, args, device)
    neg_emb, neg_mask = neg_emb.cpu(), neg_mask.cpu()
    row_embeds = {}
    for pos, idx in enumerate(target_rows, start=1):
        fc = gs.ROW_FRAME_COUNTS[idx]
        ref = gs.pack_reference(cell, fc)
        prompt = gs.make_prompt(idx, fc)
        emb, mask = gs.encode_instruction(enc_pipe, [prompt], [gs.black_composite(ref)], args, device)
        row_embeds[idx] = (emb.cpu(), mask.cpu())
        print(f'[low-vram]   encoded prompt {pos}/{len(target_rows)} (row {idx + 1:02d}/11: {gs.ROW_NAMES[idx]})', flush=True)
    del enc_pipe, mllm, processor
    gc.collect()
    torch.cuda.empty_cache()

    print('[low-vram] Stage 2/3: Encoding reference latents with RGBA VAE...', flush=True)
    vae = AutoencoderKL.from_pretrained(args.model, subfolder='vae', torch_dtype=dtype).to(device=device, dtype=dtype)
    vae.eval()
    for p in vae.parameters():
        p.requires_grad_(False)
    ref_latents_by_count = {}
    for fc in sorted(set(gs.ROW_FRAME_COUNTS[idx] for idx in target_rows)):
        ref = gs.pack_reference(cell, fc)
        ref_latents_by_count[fc] = gs.encode_reference(vae, [ref], device, dtype).cpu()
    vae.to('cpu')
    gc.collect()
    torch.cuda.empty_cache()

    print('[low-vram] Stage 3/3: Loading Boogu transformer + LoRA...', flush=True)
    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(args.base_model, subfolder='scheduler')
    transformer = BooguImageTransformer2DModel.from_pretrained(args.base_model, subfolder='transformer', torch_dtype=dtype)
    transformer.load_lora_adapter(str(gs.resolve_lora(args.model)), prefix='transformer', adapter_name='spritesheet', hotswap=False)
    transformer.set_adapter('spritesheet')
    transformer.eval()
    for p in transformer.parameters():
        p.requires_grad_(False)

    pipe = BooguImagePipeline(transformer=transformer, vae=vae, scheduler=scheduler, mllm=None, processor=None)
    pipe.text_instruction_rewriter = None
    pipe.instruction_rewriter_processor = None
    try:
        pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass

    orig_decode = vae.decode
    offload_mode = False
    def staged_decode(z, return_dict=False, **kw):
        swapped = False
        try:
            vae.to(device=z.device, dtype=z.dtype)
        except torch.OutOfMemoryError:
            swapped = True
            if not offload_mode:
                transformer.to('cpu')
            torch.cuda.empty_cache()
            vae.to(device=z.device, dtype=z.dtype)
        with torch.no_grad():
            out = orig_decode(z, return_dict=return_dict, **kw)
        vae.to('cpu')
        torch.cuda.empty_cache()
        if swapped and not offload_mode:
            transformer.to(device=device, dtype=dtype)
        return out
    vae.decode = staged_decode

    freqs_cis = _move_to_device(
        transformer.rope_embedder.get_freqs_cis(
            transformer.config.axes_dim_rope,
            transformer.config.axes_lens,
            theta=10000,
        ),
        device,
    )

    if total_gib >= 20.0:
        try:
            transformer.to(device=device, dtype=dtype)
        except torch.OutOfMemoryError:
            print('[low-vram] Full transformer on GPU hit OOM; switching to block CPU offload...', flush=True)
            offload_mode = True
            _enable_block_offload(transformer, device)
    else:
        print('[low-vram] <20 GiB VRAM detected; using transformer block CPU offload...', flush=True)
        offload_mode = True
        _enable_block_offload(transformer, device)

    generated = [None] * len(gs.ROW_FRAME_COUNTS)
    pending = list(target_rows)
    cursor = 0
    active_batch_size = min(args.row_batch_size, len(pending))
    neg_e_gpu = neg_emb.to(device)
    neg_m_gpu = neg_mask.to(device)
    while cursor < len(pending):
        indices = pending[cursor : cursor + active_batch_size]
        batch_size = len(indices)
        ref_lats = [ref_latents_by_count[gs.ROW_FRAME_COUNTS[idx]].to(device=device, dtype=dtype) for idx in indices]
        ref_sample = ref_lats[0]
        latent_shape = (int(transformer.config.in_channels), ref_sample.shape[-2], ref_sample.shape[-1])
        noise = [
            torch.randn(latent_shape, generator=torch.Generator(device=device).manual_seed(args.seed + idx + 1), device=device, dtype=dtype)
            for idx in indices
        ]
        embs = _pad_and_cat([row_embeds[idx][0] for idx in indices], dim=0).to(device)
        masks = _pad_and_cat([row_embeds[idx][1] for idx in indices], dim=0).to(device)
        pipe._text_guidance_scale = args.text_guidance
        pipe._image_guidance_scale = args.image_guidance
        pipe._empty_instruction_guidance_scale = 0.0
        pipe._cfg_range = (0.0, 1.0)
        try:
            outputs = pipe.processing(
                latents=torch.stack(noise),
                ref_latents=[[rl[0]] for rl in ref_lats],
                instruction_embeds=embs,
                freqs_cis=freqs_cis,
                negative_instruction_embeds=neg_e_gpu.expand(batch_size, -1, -1),
                instruction_attention_mask=masks,
                negative_instruction_attention_mask=neg_m_gpu.expand(batch_size, -1),
                num_inference_steps=args.steps,
                timesteps=None,
                device=device,
                dtype=dtype,
                verbose=False,
                step_func=None,
                empty_instruction_embeds=None,
                empty_instruction_attention_mask=None,
                use_empty_neg_instruct_4_ref_img_pred_at_image_guide_in_double_guide=False,
                use_empty_neg_instruct_4_ref_img_pred_at_text_guide_in_double_guide=False,
                use_boosted_orthogonal_guidance=False,
            ).detach().float().cpu()
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if active_batch_size > 1:
                active_batch_size = max(1, active_batch_size // 2)
                print(f'[low-vram] CUDA OOM: retrying with batch size {active_batch_size}', flush=True)
                continue
            if not offload_mode:
                print('[low-vram] CUDA OOM during denoising on L4; switching to transformer block CPU offload...', flush=True)
                offload_mode = True
                _enable_block_offload(transformer, device)
                continue
            raise
        for idx, output in zip(indices, outputs):
            generated[idx] = gs.tensor_to_rgba(output)
            print(f'Generated row {idx + 1:02d}/11: {gs.ROW_NAMES[idx]}', flush=True)
        cursor += len(indices)

    blank_row = Image.new('RGBA', gs.CANVAS_SIZE, (0, 0, 0, 0))
    sheet = gs.assemble_sheet([img if img is not None else blank_row for img in generated])
    if not args.keep_soft_alpha:
        sheet = gs.threshold_alpha(sheet, args.alpha_threshold)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(args.output, format='PNG', optimize=False)
    print(f'Saved {sheet.width}x{sheet.height} RGBA sprite sheet to {args.output}', flush=True)

if __name__ == '__main__':
    main()
'''
with open(LOW_VRAM_RUNNER, 'w', encoding='utf-8') as f:
    f.write(LOW_VRAM_CODE)

try:
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

run_env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
if FAST_2ROW_MODE:
    run_env['CODEX_ROWS'] = '1,3' if ACTION_ROW == 'wave' else '1,4'
else:
    run_env.pop('CODEX_ROWS', None)

for i, (key, entry) in enumerate(selected, start=1):
    out_sheet = os.path.join(slicer.OUT_DIR, f'{key}.png')
    drive_sheet = os.path.join(SHEETS_DRIVE_DIR, f'{key}.png')
    if not OVERWRITE_EXISTING and slicer.validate_sheet(out_sheet)[0]:
        print(f'[{i}/{len(selected)}] SKIP {key} (already valid)')
        continue

    src_png = os.path.join(WORK_DIR, 'assets', 'static_all_54', f'{key}.png')
    clean_png = os.path.join(CLEAN_DRIVE_DIR, f'{key}_clean.png')
    atlas_png = os.path.join(RAW_DRIVE_DIR, f'{key}_codex_atlas.png')

    clean_static_rgba(src_png, clean_png)
    print(f'[{i}/{len(selected)}] Cleaned RGBA input (mist removed) -> {clean_png}')
    display(preview_on_checker(clean_png, max_w=256))

    if REUSE_EXISTING_ATLAS and os.path.isfile(atlas_png):
        print(f'[{i}/{len(selected)}] Reusing existing Codex atlas -> {atlas_png}')
    else:
        print(f'[{i}/{len(selected)}] Running Codex-Pets-Sprite-Sheet-Generator (--row-batch-size {ROW_BATCH_SIZE})...')
        cmd = [
            sys.executable,
            LOW_VRAM_RUNNER,
            clean_png,
            '--output', atlas_png,
            '--row-batch-size', str(ROW_BATCH_SIZE),
        ]
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f'generate_sprite_sheet.py failed with exit code {rc} (see output above)')

    extract_8_from_codex_atlas(atlas_png, out_sheet, action_row=ACTION_ROW)
    shutil.copy2(out_sheet, drive_sheet)

    print(f'Full Codex Atlas ({atlas_png}):')
    display(preview_on_checker(atlas_png, max_w=768))
    print(f'Packed 512x256 Game Sheet ({drive_sheet}):')
    display(preview_on_checker(out_sheet, max_w=512))

pack_zip = os.path.join(DRIVE_KIT_DIR, 'all_sheets_pack.zip')
with zipfile.ZipFile(pack_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SHEETS_DRIVE_DIR)):
        if fn.endswith('.png'):
            zf.write(os.path.join(SHEETS_DRIVE_DIR, fn), arcname=fn)
print(f'Updated bundle: {pack_zip}')
slicer.audit_sheets()